# BioMini Phase I Educational Materials

## Notebook 04 — Separating Domain Objects and Analysis
### Domain Object와 Analyzer Architecture의 분리

Notebook 03에서는 DNA와 RNA가 biological transformation을 수행하도록 확장했습니다.

이제 다음 질문을 다룹니다.

> biological object 안에 분석 기능을 계속 추가해도 되는가?

처음에는 편해 보이지만, `Protein`에 molecular weight, hydrophobicity, composition,
feature generation까지 모두 넣기 시작하면 domain object가 너무 많은 책임을 가지게 됩니다.

이번 Notebook에서는 이 문제를 직접 경험한 뒤,
**domain object와 analysis logic을 분리하는 Analyzer architecture**를 설계합니다.

이 Notebook은 BioMini Phase I의 **Step 5 — Biological Analysis**와
**Step 6 — Analyzer Architecture**를 교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. domain object와 analysis responsibility를 구분한다.
2. 분석 기능을 domain class 안에 계속 넣을 때 생기는 문제를 설명한다.
3. `has-a`와 `is-a` 관계를 구분한다.
4. inheritance와 composition의 용도를 구분한다.
5. `SequenceAnalyzer`, `NucleicAcidAnalyzer`, `DNAAnalyzer`, `RNAAnalyzer`, `ProteinAnalyzer` 구조를 설계한다.
6. sequence composition, motif search, GC content를 구현한다.
7. protein molecular weight와 hydrophobic fraction을 구현한다.
8. empty Protein edge case를 안전하게 처리한다.
9. canonical BioMini Analyzer API와 교육용 구현을 비교한다.

## 1. Starting Point — Domain Object가 너무 커지는 문제

먼저 아주 단순한 `Protein` class에 분석 method를 직접 넣어 보겠습니다.

In [ ]:
class ValidationError(ValueError):
    pass


class Protein:
    valid_codes = set("ARNDCQEGHILKMFPSTWYV")

    def __init__(self, name: str, sequence: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        self.name = name.strip()
        self.sequence = "".join(sequence.split()).upper()

        invalid_codes = set(self.sequence) - self.valid_codes
        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

    def __len__(self) -> int:
        return len(self.sequence)

    def composition(self) -> dict[str, int]:
        counts = {}
        for code in self.sequence:
            counts[code] = counts.get(code, 0) + 1
        return counts

    def hydrophobic_fraction(self) -> float:
        hydrophobic_codes = set("AILMFWVY")

        if not self.sequence:
            return 0.0

        count = sum(
            1
            for code in self.sequence
            if code in hydrophobic_codes
        )
        return count / len(self.sequence)

In [ ]:
protein = Protein(
    "example",
    "ACDEFGHIKLMNPQRSTVWY"
)

print("length:", len(protein))
print("composition:", protein.composition())
print("hydrophobic fraction:", protein.hydrophobic_fraction())

이 방식은 작은 예제에서는 충분히 동작합니다.

하지만 앞으로 다음 method들이 계속 추가된다고 생각해 봅니다.

```text
Protein
├── composition()
├── find_motif()
├── molecular_weight()
├── hydrophobic_fraction()
├── amino_acid_fraction()
├── to_features()
├── ...
```

그러면 `Protein`이 다음 책임을 동시에 갖게 됩니다.

```text
biological identity
sequence validation
biological transformation
analysis
statistics
feature engineering
```

이것이 바로 **responsibility overload**입니다.

## 2. Separation of Concerns

BioMini에서는 다음과 같이 책임을 분리합니다.

```text
Protein
= biological domain object

ProteinAnalyzer
= Protein을 대상으로 계산과 분석을 수행하는 object
```

관계는 다음과 같습니다.

```text
ProteinAnalyzer has-a Protein
```

즉 `ProteinAnalyzer`는 `Protein`을 상속하는 것이 아니라,
`Protein` object를 내부에 가지고 사용합니다.

## 3. 최소 Domain Object 준비

이번 Notebook에서는 분석 architecture에 집중하기 위해
필요한 domain hierarchy만 간단히 구성합니다.

In [ ]:
class SequenceMolecule:
    allow_empty_sequence = False

    def __init__(self, name: str, sequence: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        normalized = "".join(sequence.split()).upper()

        if not normalized and not self.allow_empty_sequence:
            raise ValidationError("sequence must be a non-empty string")

        self.name = name.strip()
        self.sequence = normalized

    def get_sequence(self) -> str:
        return self.sequence

    def __len__(self) -> int:
        return len(self.sequence)

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"name='{self.name}', length={len(self)})"
        )


class NucleicAcid(SequenceMolecule):
    pass


class DNA(NucleicAcid):
    valid_codes = set("ATGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        invalid_codes = set(self.sequence) - self.valid_codes
        if invalid_codes:
            raise ValidationError(
                f"invalid DNA code(s): {sorted(invalid_codes)}"
            )


class RNA(NucleicAcid):
    valid_codes = set("AUGC")

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)
        invalid_codes = set(self.sequence) - self.valid_codes
        if invalid_codes:
            raise ValidationError(
                f"invalid RNA code(s): {sorted(invalid_codes)}"
            )

Protein molecular weight 계산에 필요한 residue mass도 준비합니다.

In [ ]:
class AminoAcid:
    mass_dict = {
        "A": 71.0779, "R": 156.1857, "N": 114.1026, "D": 115.0874,
        "C": 103.1429, "Q": 128.1292, "E": 129.1140, "G": 57.0513,
        "H": 137.1393, "I": 113.1576, "L": 113.1576, "K": 128.1723,
        "M": 131.1960, "F": 147.1738, "P": 97.1152, "S": 87.0773,
        "T": 101.1039, "W": 186.2099, "Y": 163.1732, "V": 99.1310,
    }

    acceptable_codes = set(mass_dict)

    def __init__(self, code: str):
        normalized = code.upper()

        if normalized not in self.acceptable_codes:
            raise ValidationError(
                f"invalid amino acid: {normalized}"
            )

        self.code = normalized

    def get_mass(self) -> float:
        return self.mass_dict[self.code]


class Protein(SequenceMolecule):
    allow_empty_sequence = True
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)

        invalid_codes = set(self.sequence) - self.valid_codes
        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

        self.amino_acids = [
            AminoAcid(code)
            for code in self.sequence
        ]

## 4. SequenceAnalyzer — 공통 분석 기능

DNA, RNA, Protein은 모두 sequence를 가지고 있으므로,
sequence level의 공통 분석은 하나의 analyzer에서 처리할 수 있습니다.

첫 번째 abstraction:

```text
SequenceAnalyzer
```

이 analyzer는 `SequenceMolecule`을 받습니다.

In [ ]:
class SequenceAnalyzer:
    def __init__(
        self,
        molecule: SequenceMolecule
    ):
        self.molecule = molecule

    def length(self) -> int:
        return len(self.molecule)

    def composition(self) -> dict[str, int]:
        counts = {}

        for code in self.molecule.sequence:
            counts[code] = (
                counts.get(code, 0)
                + 1
            )

        return counts

    def find_motif(
        self,
        motif: str
    ) -> int:
        return (
            self.molecule.sequence
            .find(motif.upper())
        )

In [ ]:
dna = DNA(
    "dna1",
    "ATGCGCGT"
)

analyzer = SequenceAnalyzer(dna)

print("length:", analyzer.length())
print("composition:", analyzer.composition())
print("motif CG:", analyzer.find_motif("CG"))
print("motif AAA:", analyzer.find_motif("AAA"))

여기서 중요한 점은 `DNA` 자체가 `composition()`을 알 필요가 없다는 것입니다.

```text
DNA
= 자신의 identity와 sequence를 유지

SequenceAnalyzer
= 그 sequence를 어떻게 분석할지 담당
```

이렇게 하면 분석 기능을 domain model과 독립적으로 확장할 수 있습니다.

## 5. NucleicAcidAnalyzer — GC Content

DNA와 RNA는 nucleic acid이므로 GC content라는 공통 분석을 가질 수 있습니다.

따라서 analyzer hierarchy도 domain hierarchy와 일부 대응합니다.

```text
SequenceAnalyzer
    ↓
NucleicAcidAnalyzer
```

In [ ]:
class NucleicAcidAnalyzer(
    SequenceAnalyzer
):
    def gc_content(self) -> float:
        sequence = self.molecule.sequence

        gc_count = (
            sequence.count("G")
            + sequence.count("C")
        )

        return (
            gc_count
            / len(sequence)
        )

In [ ]:
dna = DNA(
    "gc_example",
    "ATGCGC"
)

analyzer = NucleicAcidAnalyzer(dna)

print(
    "GC content:",
    analyzer.gc_content()
)

예를 들어:

```text
ATGCGC
```

총 길이는 6이고 G/C가 4개이므로:

```text
GC content = 4 / 6
```

입니다.

## 6. DNAAnalyzer와 RNAAnalyzer

현재 Phase I에서는 DNA와 RNA에 별도의 추가 분석 기능이 많지 않습니다.

그래도 type-level specialization을 명확히 하기 위해 다음 class를 둡니다.

In [ ]:
class DNAAnalyzer(
    NucleicAcidAnalyzer
):
    pass


class RNAAnalyzer(
    NucleicAcidAnalyzer
):
    pass

In [ ]:
dna = DNA(
    "dna2",
    "ATGC"
)

rna = RNA(
    "rna2",
    "AUGC"
)

dna_analyzer = DNAAnalyzer(dna)
rna_analyzer = RNAAnalyzer(rna)

print(
    "DNA GC:",
    dna_analyzer.gc_content()
)

print(
    "RNA GC:",
    rna_analyzer.gc_content()
)

## 7. ProteinAnalyzer

이제 Protein-specific analysis를 별도 class로 구성합니다.

```text
SequenceAnalyzer
    ↓
ProteinAnalyzer
```

여기에는 다음 기능을 넣습니다.

```text
amino-acid composition
hydrophobic fraction
molecular weight
```

In [ ]:
class ProteinAnalyzer(
    SequenceAnalyzer
):
    hydrophobic_codes = set(
        "AILMFWVY"
    )

    def amino_acid_composition(
        self
    ) -> dict[str, int]:
        return self.composition()

    def hydrophobic_fraction(
        self
    ) -> float:
        sequence = self.molecule.sequence

        if not sequence:
            return 0.0

        count = sum(
            1
            for code in sequence
            if code in self.hydrophobic_codes
        )

        return (
            count
            / len(sequence)
        )

    def molecular_weight(
        self
    ) -> float:
        if not self.molecule.amino_acids:
            return 0.0

        return (
            18.0153
            + sum(
                amino_acid.get_mass()
                for amino_acid
                in self.molecule.amino_acids
            )
        )

## 8. Amino-Acid Composition

먼저 composition을 확인합니다.

In [ ]:
protein = Protein(
    "protein1",
    "AACDE"
)

analyzer = ProteinAnalyzer(
    protein
)

print(
    analyzer.amino_acid_composition()
)

예:

```text
AACDE
```

의 경우 `A`는 2개이고 나머지는 각각 1개입니다.

`ProteinAnalyzer`는 별도의 composition logic을 다시 만들지 않고
`SequenceAnalyzer.composition()`을 재사용합니다.

## 9. Hydrophobic Fraction

Phase I에서는 다음 residue들을 hydrophobic group으로 사용합니다.

```text
A I L M F W V Y
```

따라서:

```text
hydrophobic fraction
=
hydrophobic residue count / total residue count
```

입니다.

In [ ]:
protein = Protein(
    "hydrophobic_example",
    "AILMFWVY"
)

analyzer = ProteinAnalyzer(
    protein
)

print(
    analyzer.hydrophobic_fraction()
)

In [ ]:
protein = Protein(
    "mixed_example",
    "ACDEFGHIKLMNPQRSTVWY"
)

analyzer = ProteinAnalyzer(
    protein
)

print(
    "hydrophobic fraction:",
    analyzer.hydrophobic_fraction()
)

여기서 중요한 점은 hydrophobic amino-acid set 자체도
하나의 명시적인 scientific assumption이라는 것입니다.

다른 분석 목적에서는 residue grouping이 달라질 수 있습니다.

따라서 이런 기준은 숨겨진 magic rule이 아니라 코드에서 명시적으로 보여야 합니다.

## 10. Protein Molecular Weight

Protein molecular weight를 계산할 때 각 residue mass를 더한 뒤
terminal water의 평균 질량을 추가합니다.

Phase I canonical formula:

```text
molecular weight
=
sum(residue masses)
+
18.0153
```

단, empty Protein은 별도의 정책을 사용합니다.

In [ ]:
protein = Protein(
    "small_protein",
    "ACD"
)

analyzer = ProteinAnalyzer(
    protein
)

print(
    "molecular weight:",
    analyzer.molecular_weight()
)

직접 계산해 봅니다.

In [ ]:
residue_sum = (
    AminoAcid("A").get_mass()
    + AminoAcid("C").get_mass()
    + AminoAcid("D").get_mass()
)

manual_mass = (
    residue_sum
    + 18.0153
)

print(
    "manual:",
    manual_mass
)

print(
    "analyzer:",
    analyzer.molecular_weight()
)

## 11. Empty Protein Edge Case

Notebook 03에서 stop codon만 있는 RNA가
valid empty Protein을 생성할 수 있음을 확인했습니다.

따라서 다음은 valid object입니다.

```python
Protein("empty", "")
```

이때 분석 결과도 정의해야 합니다.

In [ ]:
empty_protein = Protein(
    "empty",
    ""
)

empty_analyzer = ProteinAnalyzer(
    empty_protein
)

print(
    "length:",
    empty_analyzer.length()
)

print(
    "composition:",
    empty_analyzer.composition()
)

print(
    "hydrophobic fraction:",
    empty_analyzer.hydrophobic_fraction()
)

print(
    "molecular weight:",
    empty_analyzer.molecular_weight()
)

Phase I canonical policy:

```text
empty Protein length                 = 0
empty Protein composition            = {}
empty Protein hydrophobic fraction   = 0.0
empty Protein molecular weight       = 0.0
```

여기서 water mass `18.0153`을 반환하지 않는다는 점이 중요합니다.

empty translation product는 실제 peptide chain이 존재하지 않는 상태이므로
Phase I에서는 molecular weight를 `0.0`으로 정의합니다.

## 12. has-a vs is-a

이제 두 관계를 비교해 봅니다.

### Domain inheritance

```text
DNA is-a NucleicAcid
RNA is-a NucleicAcid
Protein is-a SequenceMolecule
```

이 관계는 자연스럽습니다.

### Analyzer composition

```text
ProteinAnalyzer has-a Protein
DNAAnalyzer has-a DNA
```

분석기는 biological object 자체가 아닙니다.

따라서 다음 설계는 피합니다.

```text
ProteinAnalyzer is-a Protein
```

대신:

```python
ProteinAnalyzer(protein)
```

형태를 사용합니다.

## 13. Analyzer Hierarchy 정리

현재 구조:

```text
SequenceAnalyzer
├── NucleicAcidAnalyzer
│   ├── DNAAnalyzer
│   └── RNAAnalyzer
└── ProteinAnalyzer
```

Domain side:

```text
SequenceMolecule
├── NucleicAcid
│   ├── DNA
│   └── RNA
└── Protein
```

두 hierarchy는 서로 관련되지만 같은 hierarchy는 아닙니다.

```text
Domain
= what the biological object is

Analyzer
= how the object is analyzed
```

## 14. Self-Check

In [ ]:
dna = DNA(
    "dna_test",
    "ATGCGC"
)

dna_analyzer = DNAAnalyzer(
    dna
)

assert dna_analyzer.length() == 6
assert dna_analyzer.composition()["G"] == 2
assert abs(
    dna_analyzer.gc_content()
    - (4 / 6)
) < 1e-12

protein = Protein(
    "protein_test",
    "AILM"
)

protein_analyzer = ProteinAnalyzer(
    protein
)

assert (
    protein_analyzer.length()
    == 4
)

assert (
    protein_analyzer.hydrophobic_fraction()
    == 1.0
)

assert (
    protein_analyzer.molecular_weight()
    > 0.0
)

empty = Protein(
    "empty",
    ""
)

empty_analyzer = ProteinAnalyzer(
    empty
)

assert (
    empty_analyzer.molecular_weight()
    == 0.0
)

assert (
    empty_analyzer.hydrophobic_fraction()
    == 0.0
)

print(
    "Notebook 04 self-check: PASS"
)

## 15. Canonical BioMini와 비교

실제 BioMini package가 설치되어 있다면 같은 Analyzer architecture를 사용할 수 있습니다.

In [ ]:
try:
    from biomini import (
        DNA as BioMiniDNA,
        Protein as BioMiniProtein,
        DNAAnalyzer as BioMiniDNAAnalyzer,
        ProteinAnalyzer as BioMiniProteinAnalyzer,
    )

    bm_dna = BioMiniDNA(
        "dna1",
        "ATGCGC"
    )

    bm_dna_analyzer = (
        BioMiniDNAAnalyzer(
            bm_dna
        )
    )

    bm_protein = BioMiniProtein(
        "protein1",
        "ACDEFG"
    )

    bm_protein_analyzer = (
        BioMiniProteinAnalyzer(
            bm_protein
        )
    )

    print(
        "Canonical DNA GC:",
        bm_dna_analyzer.gc_content()
    )

    print(
        "Canonical protein MW:",
        bm_protein_analyzer.molecular_weight()
    )

    print(
        "Canonical hydrophobic fraction:",
        bm_protein_analyzer.hydrophobic_fraction()
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

## 16. Canonical BioMini Analyzer의 추가 기능

현재 production `ProteinAnalyzer`에는 이번 Notebook에서 배운 기능 외에도
다음 기능이 있습니다.

```text
amino_acid_fraction_features()
to_features()
```

이 기능들은 Protein 분석 결과를 machine-learning representation으로 바꾸는 역할을 합니다.

하지만 여기서는 일부러 아직 구현하지 않습니다.

왜냐하면 다음 Notebook의 핵심 질문이 바로 이것이기 때문입니다.

> biological analysis 결과를 어떻게 machine-learning feature로 표현할 것인가?

## 17. Domain Object와 Analysis Object를 분리했을 때의 장점

### 1. Domain object가 작고 명확해진다

```text
Protein
= Protein이라는 biological entity
```

### 2. 분석 기능을 독립적으로 확장할 수 있다

```text
ProteinAnalyzer
ExperimentalProteinAnalyzer
StructuralProteinAnalyzer
```

같은 별도 분석기를 만들 수 있습니다.

### 3. 동일 object에 여러 분석 전략을 적용할 수 있다

```text
Protein
 ├── analyzer A
 ├── analyzer B
 └── analyzer C
```

### 4. ML representation과도 분리하기 쉬워진다

```text
Protein
 ↓
ProteinAnalyzer
 ↓
FeatureSet
```

이 구조가 Notebook 05로 연결됩니다.

## 18. What We Learned

이번 Notebook의 핵심 흐름:

```text
fat domain class
        ↓
responsibility overload
        ↓
separation of concerns
        ↓
SequenceAnalyzer
        ↓
NucleicAcidAnalyzer
        ↓
DNAAnalyzer / RNAAnalyzer
        ↓
ProteinAnalyzer
        ↓
composition over unnecessary inheritance
```

가장 중요한 설계 원칙:

> biological object는 자신이 무엇인지 표현하고,
> Analyzer는 그 object를 어떻게 계산하고 해석할지를 담당한다.

## 19. Bridge to Notebook 05

이제 ProteinAnalyzer를 통해 다음 값을 얻을 수 있습니다.

```text
length
molecular weight
hydrophobic fraction
amino-acid composition
```

하지만 machine-learning model은 `Protein` object 자체를 직접 사용하지 않습니다.

따라서 다음 단계가 필요합니다.

```text
Protein
 ↓
ProteinAnalyzer
 ↓
numerical representation
```

Notebook 05에서는 이를 위해:

```text
FeatureSet
Sample
Dataset
```

을 도입합니다.

즉 다음 주제는:

> **From Biology to Feature Engineering**

입니다.

## 20. Exercises

### Exercise 1 — Sequence Composition
다음 DNA sequence의 composition을 계산하십시오.

```text
AAATTTGGCC
```

직접 센 값과 `SequenceAnalyzer.composition()` 결과를 비교하십시오.

### Exercise 2 — Motif Search
다음 sequence에서 `"GCG"` motif의 첫 위치를 찾으십시오.

```text
ATGCGCGT
```

motif가 없을 때 반환값도 확인하십시오.

### Exercise 3 — GC Content
다음 DNA의 GC content를 계산하십시오.

```text
ATATGCGC
```

### Exercise 4 — Hydrophobic Fraction
다음 Protein을 비교하십시오.

```text
AILMFWVY
DEKRQN
ACDEFGHIKLMNPQRSTVWY
```

어떤 sequence가 가장 높은 hydrophobic fraction을 가지는지 확인하십시오.

### Exercise 5 — Molecular Weight
`ACD`의 molecular weight를 residue mass와 water mass를 이용해 직접 계산하고
`ProteinAnalyzer.molecular_weight()`와 비교하십시오.

### Exercise 6 — Empty Protein
empty Protein에 대해 다음 값을 확인하십시오.

```text
length
composition
hydrophobic fraction
molecular weight
```

왜 각 값이 그렇게 정의되는지 설명하십시오.

### Exercise 7 — Design Exercise
`GC content`를 `DNA` class 안에 넣는 설계와
`DNAAnalyzer` 안에 넣는 설계의 장단점을 비교하십시오.

### Exercise 8 — Architecture Exercise
다음 중 inheritance와 composition 중 어느 것이 더 적합한지 판단하고 이유를 설명하십시오.

```text
DNA → NucleicAcid
ProteinAnalyzer → Protein
RNAAnalyzer → SequenceAnalyzer
```